# LLM Client — Gemma 4 E2B on EKS

Uses the **Strands Agents SDK with LiteLLM** to talk to the Gemma 4 E2B model
running on your EKS cluster via LLMKube.  
No OpenAI account or API key required — the model runs entirely on your own AWS infrastructure.

In [ ]:
# Install dependencies
!pip install 'strands-agents[litellm]' litellm --quiet

In [ ]:
import subprocess
from strands import Agent
from strands.models.litellm import LiteLLMModel

In [ ]:
# ── Auto-detect the Load Balancer URL from your EKS cluster ──────────────────
# Requires kubectl to be configured:
#   aws eks update-kubeconfig --region us-west-2 --name llm-eks

result = subprocess.run(
    [
        "kubectl", "get", "svc", "gemma-e2b-service",
        "-o", "jsonpath={.status.loadBalancer.ingress[0].hostname}"
    ],
    capture_output=True, text=True
)

LB_HOSTNAME = result.stdout.strip()

if not LB_HOSTNAME:
    raise RuntimeError(
        "Could not detect Load Balancer hostname.\n"
        "Make sure kubectl is configured and the service is running:\n"
        "  aws eks update-kubeconfig --region us-west-2 --name llm-eks\n"
        "  kubectl get svc gemma-e2b-service"
    )

API_BASE = f"http://{LB_HOSTNAME}:8080"
print(f"✅ Load Balancer detected: {API_BASE}")

In [ ]:
# ── Configure the model via LiteLLM ──────────────────────────────────────────
# LiteLLM's openai/ prefix lets us talk to any OpenAI-compatible endpoint.
# LLMKube exposes exactly this API — no real API key needed.

model = LiteLLMModel(
    model_id="openai/gemma-e2b",   # openai/ prefix = OpenAI-compatible endpoint
    client_args={
        "api_key": "not-needed",   # LLMKube has no auth
        "api_base": API_BASE,
        "timeout": 300,            # 5 min — CPU inference is slow (~6 tok/s)
    },
    params={
        # Gemma 4 E2B is a reasoning model — it uses tokens for internal
        # thinking before writing the answer. Use 800+ to get full responses.
        "max_tokens": 800,
        "temperature": 0.7,
    }
)

In [ ]:
# ── Create the Strands agent ──────────────────────────────────────────────────
agent = Agent(
    model=model,
    system_prompt="You are a helpful assistant powered by Gemma 4 E2B running on AWS EKS via LLMKube."
)

In [ ]:
# ── Ask a question ────────────────────────────────────────────────────────────
# On CPU (t3.xlarge) expect 60-120 seconds per response.
print("⏳ Sending request... (60-120 seconds on CPU)\n")

response = agent("What is Kubernetes?")
print(response)

In [ ]:
# ── Multi-turn conversation ───────────────────────────────────────────────────
# The agent keeps conversation history automatically.
print("⏳ Follow-up question...\n")

response2 = agent("What is the capital of India? Give me 3 fun facts about it.")
print(response2)